In [49]:
import math
import random

In [50]:
class Mathlib:
    def randn(*shape):
        pass

    def clip(val, minVal=0, maxVal=1):
        return(min(maxVal, max(minVal, val)))

    def mean(vals):
        return(sum(vals)/len(vals))
    
    def argmax(vals):
        maxVal = vals[0]
        maxIndex = 0
        for i, v in enumerate(vals):
            if v > maxVal:
                maxVal = v
                maxIndex = i
        return(maxIndex)
    
    def argmin(vals):
        minVal = vals[0]
        minIndex = 0
        for i, v in enumerate(vals):
            if v < minVal:
                minVal = v
                minIndex = i
        return(minIndex)

    def dot2Vectors(vector1, vector2):
        out = 0
        for v1, v2 in zip(vector1, vector2):
            out += v1*v2
        return(out)

    def normalize(values):
        out = []
        total = sum(values)
        for v in values:
            out.append(v/total)
        return(out)

#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<p float="left">
  <img src="proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1" alt="Softmax Proof" width="500"  style="vertical-align: top;">
  <img src="proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1" alt="ReLU Proof" width="500"  style="vertical-align: top;">
</p>


In [ ]:
class Activation:
    class ReLU:
        """ Returns 0 if values and x if its not """
        @staticmethod
        def forward(val):
            return(max(0, val))    #< by clipping 0, you de-linearize your data
        
        @staticmethod
        def backward(val):
            """ partial derivative of max(x,0) with respect to x """
            return(1 if val > 0 else 0)

    class Pass:
        """ returns x """
        @staticmethod
        def forward(val):
            return(val)

        @staticmethod
        def backward(*args):
            """ derivative of x with respect to x """
            return(1)

    class Softmax:
        """ converts any output to be squashed from 0 to 1 and also had a nice derivative when paired with  Cross-Entropy """
        @staticmethod
        def forward(vals):
            vals = [math.e**val for val in vals]
            vals = Mathlib.normalize(vals)
            return(vals)
        
        @staticmethod
        def backward(vals):
            return(Activation.Softmax.backward_jacobian(vals))

        @staticmethod
        def backward_jacobian(vals):
            """ derivative of e^x/(sum(e^x) for x in vals) with respect to x
            For proof, see image above or in 'proofs_math/ActivationFuncs'"""
            jacobian = []
            for i, val in enumerate(vals):
                row = []
                for j in range(len(vals)):
                    if j == i:
                        row.append(val*(1-val))
                    else:
                        row.append(-val*vals[j])
                jacobian.append(row)
            return(jacobian)
        
        @staticmethod
        def backward_crossEntropy(vals):
            pass

    class ProtectedSoftmax:
        """ Softmax but without any overflow from e^x being too large """
        @staticmethod
        def forward(vals):
            """ softmax, but between 0 and 1 """
            maxVal = max(vals)
            return(Activation.Softmax.forward([val-maxVal for val in vals]))
        
        @staticmethod
        def backward(vals):
            """ derivative of e^x with respect to x as protected softmax is just regular softmax with extra steps to ensure values are normalized """
            return(Activation.Softmax.backward(vals))


In [181]:
class Loss:
    """ Calculates the error of the output (mainly Softmax) using -log(x) """
    @staticmethod
    def forward_single(val):
        return(-math.log(Mathlib.clip(val, 1e-7, 1-1e-7)))

    @staticmethod
    def forward(vals, targets):
        """calculate the cross entropy loss of a softmax output

        Args:
            vals (list): results
            targets (list): target results
        """
        out = 0
        for v, t in zip(vals, targets):
            out += t*Loss.forward_single(v)   #< loss times the weight of the loss, more important = bigger loss
        return(out/len(vals))                 #< mean of the losses

    @staticmethod
    def backward(val):
        return(-1/val)


In [180]:
class Accuracy_hard:
    """ calculates how accurate the the result is, for the hard version, it gets an array of index (must be all 0 and a single value has to be 1), and it returns 1 if that most likely output is at the correct index and 0 if its not """
    def __init__(self):
        self._length = 0
        self.correct = 0
        self.accuracy = 0.0

    @staticmethod
    def calc(result, target):
        if type(target) == list:
            target = Mathlib.argmax(target)
        if Mathlib.argmax(result) == target:
            return(1)
        return(0)

    def epoch(self, result, target):
        self.correct += Accuracy_hard.calc(result, target)
        self._length += 1
        self.accuracy = self.correct/self._length
        return(self.accuracy)

class Accuracy_soft:
    """ calculates how accurate the the result is, for the soft version, it get an array of target results, ex: [0.2, 0.8, 0.0, 0.0] and returns a number to quantify how close it is"""
    def __init__(self):
        self._length = 0
        self.correct = 0.0
        self.accuracy = 0.0

    @staticmethod
    def calc(result, target):
        dot = Mathlib.dot2Vectors(result, target)
        norm_r = math.sqrt(sum(r*r for r in result))
        norm_t = math.sqrt(sum(t*t for t in target))
        soft = dot / (norm_r * norm_t)      #< normalize between 0 and 1
        return(soft)

    def epoch(self, result, target):
        soft = Accuracy_soft.calc(result, target)
        self._length += 1
        self.correct += soft
        self.accuracy = self.correct/self._length
        return(self.accuracy)
        

In [165]:
class Neuron:
    """ A single neuron, stores its inputs, weights, bias, and activation function """
    def __init__(self, inputs, weights, bias, activationFunc=Activation.Pass):
        self.inputs = inputs      #< all outputs from previous layer
        self.weights = weights    #< one weight for every input
        self.bias = bias          #< one bias per neuron
        self.activationFunc = activationFunc

    def forward(self):
        """
        Compute the neuron's output.
        Multiply each input by its corresponding weight, sum the
        results, add the bias and then apply the activation function.
        
        multiply the inputs [i1, i2.. in] with weights [w1, w1... wn] to get i1*w1+i2*w2... +in*wn, then add the bias
        """
        
        return(self.activationFunc.forward(Mathlib.dot2Vectors(self.inputs, self.weights)+self.bias))

    def backward(self, output):
        """ Compute gradient 
        since forward is computed as Activation(sum(x1*w1, x2*w2..., bias)),
        the backward for the weights would be computed as [Activation`(...)*sum`(...)*xi] for every element,
        and the backward for the inputs would be computed as [Activation`(...)*sum`(...)*wi] for every element,
        note that sum`(...) is equal to 1 no matter the reference or input.
        """
        activation_dx = self.activationFunc.backward(output)
        inputs_dx = [activation_dx*w for w in self.weights]
        weights_dx = [activation_dx*i for i in self.inputs]
        bias_dx = activation_dx
        return(inputs_dx, weights_dx, bias_dx)


In [176]:
class Layer:
    """A collection of neurons forming a single layer in the network"""
    def __init__(self, inputs, weights, biases,
                 normalize=True,
                 activationFunc=Activation.Pass,   #< Neuron-level activation function
                 layerActivationFunc=None,         #< Layer activation func is the activation func that gets applied to every output AFTER its computed and a neuron-level activation function is already ran on it. This is useful mostly for Softmax at the last layer
                 *args, **kwargs):
        self.out = None
        self.normalize = normalize
        self.activationFunc = activationFunc
        self.layerActivationFunc = layerActivationFunc
        self.neurons = self._createNeurons(inputs, weights, biases, activationFunc=activationFunc, *args, **kwargs)  # create neuron objects

    def _createNeurons(self, inputs, weights, biases, activationFunc, *args, **kwargs):
        neurons = []
        for w, b in zip(weights, biases):
            neurons.append(Neuron(inputs, w, b, activationFunc, *args, **kwargs))
        return(neurons)
    
    def forward(self):
        """Calculate outputs for all neurons in the layer, apply layer-level
        activation and optional normalization."""
        self.out = [neuron.forward() for neuron in self.neurons]    #< saving to use in the backpropagation with chain rule
        res = self.out
        if self.layerActivationFunc:
            res = self.layerActivationFunc.forward(self.out)
        if self.normalize:
            res = Mathlib.normalize(self.out)                       #< we put this AFTER activation function because partial derivative of sum(x+c) = 1
        return(res)
    
    def backward(self, outputs):
        """Calculate gradients for all neurons in the layer."""
        neuronOutputs = [neuron.backward(output) for neuron, output in zip(self.neurons, self.out)] #< we can calculate these 
        if self.layerActivationFunc:
            res = []
            for n, l in zip(neuronOutputs, self.layerActivationFunc.backward(outputs)):
                print(n, l)
                res.append(n*l)
            return(res)
        return(neuronOutputs)

    def calcLoss(self, target):
        """Convenience wrapper for computing loss using either a hot target, or a distribution of targets"""
        if type(target) == list:
            return(Loss.crossEntropyLoss(self.out, target))
            
        return(Loss.crossEntropyLoss_single(self.out[target]))


In [167]:
class Batch:
    """ Process multiple samples at once in a batch."""
    def __init__(self, inputsBatch, *args, **kwargs):
        self.batch = self._createBatch(inputsBatch, *args, **kwargs)  # create a Layer for each sample

    def _createBatch(self, inputsBatch, *args, **kwargs):
        """Helper method to create a Layer object for each sample in the batch."""
        batch = []
        for inputs in inputsBatch:
            batch.append(Layer(inputs, *args, **kwargs))  # instantiate Layer and add to batch
        return(batch)
    
    def forward(self):
        """Run all layers in the batch and return their outputs."""
        return([layer.forward() for layer in self.batch])
    
    def calcLoss(self, correctIndexes):
        """Compute mean loss across the batch given a list of target indexes."""
        return(Mathlib.mean([self.batch[i].calcLoss(correctIndexes[i]) for i in range(len(self.batch))]))  # average of per-sample losses


In [168]:
class Train:
    """ TBD """
    def __init__(self, weights, biases, learningRate=0.01):
        self.weights = weights
        self.biases = biases
        self.learningRate = learningRate

    def step(self, inputs, correctIndex):
        """Perform a single gradient descent update for one example.

        Args:
            inputs: list of input features for the sample
            correctIndex: index of the true class in the output vector
        """
        layer = Layer(inputs, self.weights, self.biases)
        out = layer.forward()
        for i in range(len(self.weights)) :
            # target probability is 1 for correct class, 0 otherwise (eg: 0 for cat, 0 for dog, 1 for horse)
            target = 0
            if i == correctIndex:
                target = 1
  
            # protectedSoftmax produces an output of e^x/(sum(e^x))
            # the error function is -log(result)
            # the error is -sum(targetX*log(e^x/(sum(e^x)))
            # for target with 0s and 1s [0, 0, 1, 0], the error simplifies to -log(e^x/(sum(e^x))
            # to know if the error has improved, we need to take the derivative of the error
            # the derivative of -log(e^x/(sum(e^x)) simplifies to result-target
            error = out[i] - target
            for j in range(len(self.weights[i])):
                # learning rate determines the size of the change
                # the error determines how far away we are, therefore the size of the update
                # inputs determines the importance of that input
                # an smaller input that has less effect, will be changed less
                self.weights[i][j] -= self.learningRate * error * inputs[j]
            # learning rate determines the size of the change
            # the error determines how far away we are, therefore the size of the update
            # since we update both the weights and biases, we might overcompensate, which is ok because there are multiple iterations
            self.biases[i] -= self.learningRate * error

    def train(self, inputsBatch, correctIndexes, epochs=10):
        """Train for a number of epochs """
        for _ in range(epochs):
            for inputs, correct in zip(inputsBatch, correctIndexes):
                self.step(inputs, correct)

## Function that will be used to create datasets

In [169]:
def randomNumber(minimum=-10.0, maximum=10.0, decimals=2):
    return(round(random.uniform(minimum, maximum), decimals))

def createInputs(size=10):
    return([randomNumber() for _ in range(size)])

def createLayerData(size=5, inputs=10):
    weights = [[randomNumber(minimum=-1.0, maximum=1.0) for _ in range(inputs)] for _ in range(size)]
    biases = [randomNumber() for _ in range(size)]
    return(weights, biases)

## Now lets run a forward pass

In [177]:
random.seed(0)

inputs = createInputs(10)
weights1, biases1 = createLayerData(5)
targets = [0,1,0,0,0]
layer1 = Layer(inputs, weights1, biases1, activationFunc=Activation.ReLU)
layer1Output = layer1.forward()
weights2, biases2 = createLayerData(5)
layer2 = Layer(layer1Output, weights2, biases2, activationFunc=Activation.ReLU, layerActivationFunc=Activation.Softmax)
layer2Output = layer2.forward()
error = Loss.forward(layer2Output, targets)

## seeing model accuracy
accuracy = Accuracy_hard.calc(layer2Output, targets)    #< we want to maximize the 2nd output
print("INPUTS: ", inputs)
print("WEIGHTS: ", weights1)
print("WEIGHTS: ", biases1)
print("RESULT: ", layer1Output)
print()
print("WEIGHTS: ", weights2)
print("WEIGHTS: ", biases2)
print("RESULT: ", layer2Output)
print()
print()
print("ERROR: ", error)
print("ACCURACY: ", accuracy)


INPUTS:  [6.89, 5.16, -1.59, -4.82, 0.23, -1.9, 5.68, -3.93, -0.47, 1.67]
WEIGHTS:  [[0.82, 0.01, -0.44, 0.51, 0.24, -0.5, 0.82, 0.97, 0.62, 0.8], [-0.38, 0.46, 0.8, 0.37, -0.06, -0.8, -0.13, 0.22, 0.83, 0.93], [-0.05, 0.73, -0.48, 0.61, 0.1, -0.97, 0.44, -0.2, 0.65, 0.34], [-1.0, -0.01, 0.74, -0.51, -0.35, 0.74, -0.62, 0.14, -0.52, 0.94], [0.61, -0.1, -0.84, -0.36, 0.02, 0.87, -0.78, 0.1, 0.41, 0.09]]
WEIGHTS:  [6.29, 0.81, 9.28, 2.06, 1.75]
RESULT:  [0.4226635844998777, 0.0, 0.513155014101557, 0.0, 0.06418140139856537]

WEIGHTS:  [[-0.11, 0.19, -0.23, 0.15, -0.42, -0.62, -0.63, 0.23, 0.31, -0.05], [-0.82, 0.52, 0.75, 0.85, 0.68, 0.8, 0.85, 0.08, -0.22, 0.41], [-0.45, 0.62, 0.7, 0.79, 0.18, 0.9, 0.16, -0.1, 0.32, 0.99], [0.83, 0.59, -0.84, 0.23, -0.03, 0.26, 0.69, -0.51, 0.46, -0.77], [-0.56, 0.59, -0.33, 0.63, -0.8, -0.71, 0.4, -0.91, 0.15, 0.82]]
WEIGHTS:  [0.68, 3.61, -9.47, 2.7, 2.13]
RESULT:  [0.057670933388730236, 0.43583586649325207, 0.0, 0.309038317909474, 0.19745488220854357]

## Now lets run a backward pass to see our models error

In [179]:
d_error  = Loss.backward(error)
print(d_error)
d_layer2 = layer2.backward(layer2Output)
print(d_layer2)
d_layer1 = layer1.backward(layer1Output)
print(d_layer1)
# d_inputs

-6.020545282918966
([-0.11, 0.19, -0.23, 0.15, -0.42, -0.62, -0.63, 0.23, 0.31, -0.05], [0.4226635844998777, 0.0, 0.513155014101557, 0.0, 0.06418140139856537], 1) [0.05434499683080288, -0.025135061224951863, -0.0, -0.017822528246722515, -0.011387407359128491]


TypeError: can't multiply sequence by non-int of type 'list'